# Decision Tree to predict Affairs

Who has an affair? Use Decision Trees to classify individuals based on demographic and personal attributes.

In [ ]:
%pip install scikit-learn

### Step 1: Load and Explore the Data

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

#from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import plot_tree
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error, mean_squared_error, r2_score


# Import affairs.csv file
df = pd.read_csv('?.csv')

# Import affairs_occupations.csv file
df_occupations = pd.read_csv('?.csv')

# Join the two dataframes on the '?' column
df = df.merge(df_occupations, on='?', how='left')

df

### Step 2a: Predicting years of education with *regression*

Use regression to predict the number of years of education based on various features.

In [ ]:
# Use OLS regression example with statsmodels
import statsmodels.api as sm

numeric_features = ['?']
categorical_features = ['?'] 
X = df[numeric_features + categorical_features]
y = df['?']

# One-Hot Encode the categorical predictors as 1/0 variables
# Use dtype=int to ensure numeric encoding and drop_first=True to avoid multicollinearity.
# Note that we need to use drop_first=True here for OLS regression, while for other models it may not be necessary.
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True, dtype=int)

# Fit the model using OLS for comparison
X_encoded = sm.add_constant(X_encoded)  # Adds a constant term to the predictor
model_ols = sm.OLS(y, X_encoded).fit()
predictions = model_ols.predict(X_encoded)
print(model_ols.summary())



### Step 2b: Predicting years of education with *decision trees*

Use decision trees to predict the number of years of education based on various features.

In [ ]:
# Use a decision tree

# One-Hot Encode the categorical predictors as 1/0 variables
# Use dtype=int to ensure numeric encoding.
# We can use drop_first=True, but it isn't necessary for decision trees.
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=False, dtype=int)


# Create training / test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.25, random_state=42
)

# Create and fit decision tree
tree = DecisionTreeRegressor(
    random_state=42,
    max_depth=3,          # limit depth for stability and interpretability
    min_samples_leaf=5
)
tree.fit(X_train, y_train)

# Predict on train set
y_pred = tree.predict(X_train)
    
# Calculate metrics, using mae (mean absolute error), rmse (root mean squared error), r2 (R-squared)
mae = mean_absolute_error(y_train, y_pred)
rmse = mean_squared_error(y_train, y_pred)
r2 = r2_score(y_train, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R^2:", r2)

# Feature importance, which is defined as the total reduction of the criterion (MSE) brought by that feature.
feature_importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': tree.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importances)

# Plot the decision tree
plt.figure(figsize=(30,10))
plot_tree(tree, feature_names=X_encoded.columns, filled=True, rounded=True, fontsize=18)
plt.show()

### Measure accuracy on test set.



In [ ]:
# Measure accuracy on test set.
y_pred = tree.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R^2:", r2)

## Step 3a: Predict affair variable with logistic regression

In [ ]:
## Step 3b: Predict affair variable with logistic regression
import statsmodels.api as sm

numeric_features = ['?']
categorical_features = ['?'] 
X = df[numeric_features + categorical_features]
y = df['?']

# One-hot encode
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True, dtype=int)

X_encoded = sm.add_constant(X_encoded)  # Adds a constant term to the predictor
model_logistic = sm.Logit(y, X_encoded).fit()
predictions = model_logistic.predict(X_encoded)
print(model_logistic.summary())

# Calculate accuracy of predictions using a confusion matrix
predicted_classes = (predictions >= 0.5).astype(int)
conf_matrix = confusion_matrix(y, predicted_classes)
accuracy = accuracy_score(y, predicted_classes)
print("Confusion Matrix:\n", conf_matrix)
print("Accuracy:", accuracy)


### Step 3b: Predict affair variable with decision tree classifier

In [ ]:

numeric_features = ['?']
categorical_features = ['?'] 
X = df[numeric_features + categorical_features]
y = df['?']

# One-hot encode
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=False, dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.25, random_state=42
)

clf = DecisionTreeClassifier(
    random_state=42,
    max_depth=3,
    min_samples_leaf=5
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=clf.classes_, cmap='Blues')
plt.show()

# Print accuracy and confusion matrix
print("Accuracy:", acc)
print("Confusion Matrix:\n", cm)

In [ ]:

# Show decision tree. The top line shows the decision rule.
# The second line shows gini, which is defined as 1 - sum(p_i^2) for each class i, 
#   where p_i is the proportion of samples of class i at that node.
# Gini ranges from 0 (pure node) to 0.5 (impure node with equal class distribution).
# The line with value = ... shows the number of samples at that node.
# The last line shows the most common class distribution at that node.

plt.figure(figsize=(20,10))
plot_tree(
    clf,
    feature_names=X_train.columns,
    class_names=[str(c) for c in clf.classes_],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.show()